In [1]:
import os
import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd  
from shapely.geometry import Point

In [2]:
BASIN_SHP_PATH = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\data\basin_boudary\mahanadi_basin.shp"
BASE_DIR = r"C:\Users\ps302\OneDrive\Desktop\Hydrology"
DATA_DIR = os.path.join(BASE_DIR, "data")
RASTER_DIR = os.path.join(DATA_DIR, "Flood_data_final")
print(RASTER_DIR)

C:\Users\ps302\OneDrive\Desktop\Hydrology\data\Flood_data_final


In [3]:
# STEP 1: Raster Feature Map
RASTER_FEATURE_MAP = {
    "distance_to_river": os.path.join(RASTER_DIR, "distance_to_river.tif"),
    "aspect": os.path.join(RASTER_DIR, "aspect.tif"),
    "dem": os.path.join(RASTER_DIR, "dem_30.tif"),
    "flow_accumulation": os.path.join(RASTER_DIR, "flow_acc_30m.tif"),
    "twi": os.path.join(RASTER_DIR, "TWI.tif"),
    "slope": os.path.join(RASTER_DIR, "fixed_slope.tif"),
     "flood": os.path.join(RASTER_DIR, "flood_output.tif"),  
    "rainfall": os.path.join(RASTER_DIR, "rainfall_30m_f.tif"),
    "drainage_density": os.path.join(RASTER_DIR, "drainage_density_30.tif"), 
    "ext_rainfall": os.path.join(RASTER_DIR, "ext_rainfall.tif"),
    "lulc": os.path.join(RASTER_DIR, "lulc_30m.tif"),
    "soil": os.path.join(RASTER_DIR, "soil_clay_30m.tif"),
}

In [4]:
floos_path  = os.path.join(RASTER_DIR, "dem_30.tif")
with rasterio.open(floos_path) as src:
    print(src.crs)

EPSG:32645


In [5]:
# STEP 2: Generate random points using Shapefile Bounds
# Read the shapefile
basin_gdf = gpd.read_file(BASIN_SHP_PATH)

In [6]:
basin_gdf = basin_gdf.to_crs("EPSG:32645")
basin_gdf

,Basin_Code,Average_An,Average_00,Utilisable,No_of_Hydr,Storage_Co,Storage_Un,Storage_00,Area_in_Sq,STATE,Basin_Name,Total_Live,st_area_sh,st_length_,geometry
0,08,66.88,66.88,50.0,37.0,12.779,1.465,10.094,141589.0,"Madhya Pradesh, Chhattisgarh, Odisha, Jharkhan...",Mahanadi,14.244,1.396815e+11,3.326751e+06,"MULTIPOLYGON (((41057.653 2612458.456, 41440.3..."


In [1]:
# LOAD & ALIGN SHAPEFILE (Fix CRS)
print("Loading Basin Shapefile...")
basin_gdf = gpd.read_file(BASIN_SHP_PATH)

print(" Checking Raster CRS...")
# Open DEM just to get the correct UTM projection
with rasterio.open(RASTER_FEATURE_MAP["dem"]) as src:
    raster_crs = src.crs

# Reproject Shapefile to match Raster CRS (Crucial Step!)
if basin_gdf.crs != raster_crs:
    print(f" Reprojecting basin from {basin_gdf.crs} to {raster_crs}...")
    basin_gdf = basin_gdf.to_crs(raster_crs)

# Combine multi-polygons into a single geometry for checking
basin_geom = basin_gdf.geometry.unary_union
minx, miny, maxx, maxy = basin_geom.bounds

Loading Basin Shapefile...


NameError: name 'gpd' is not defined

In [8]:
basin_gdf

,Basin_Code,Average_An,Average_00,Utilisable,No_of_Hydr,Storage_Co,Storage_Un,Storage_00,Area_in_Sq,STATE,Basin_Name,Total_Live,st_area_sh,st_length_,geometry
0,08,66.88,66.88,50.0,37.0,12.779,1.465,10.094,141589.0,"Madhya Pradesh, Chhattisgarh, Odisha, Jharkhan...",Mahanadi,14.244,1.396815e+11,3.326751e+06,"MULTIPOLYGON (((41057.653 2612458.456, 41440.3..."


In [9]:

# step3: GENERATE POINTS STRICTLY INSIDE BASIN
points = []
target_points = 10000

In [12]:
print(f"Generating {target_points} points strictly INSIDE the Mahanadi boundary...")
while len(points) < target_points:
    # Create random point inside the bounding box
    x = np.random.uniform(minx, maxx)
    y = np.random.uniform(miny, maxy)
    p = Point(x, y)
    
    # MAGIC STEP: Check if point is actually inside the irregular polygon shape
    if basin_geom.contains(p):
        points.append((x, y))

print(" Points generated successfully!")

Generating 10000 points strictly INSIDE the Mahanadi boundary...
 Points generated successfully!


In [14]:
print(basin_geom.contains(p))

True


In [15]:

#  RASTER SAMPLING FUNCTION
def sample_all_rasters(x, y, rasters):
    values = {}
    for name, path in rasters.items():
        if not os.path.exists(path):
            print(f" Warning: Missing {name} at {path}")
            continue 
            
        with rasterio.open(path) as src:
            val = list(src.sample([(x, y)]))[0][0]
            
            # Skip invalid pixels (NoData values, outside actual raster data)
            if not np.isfinite(val) or val == src.nodata:
                return None  
                
            values[name] = val
            
    return values

In [16]:

#  BUILD DATASET
data = []
point_id = 0

print(" Sampling raster data for each point... This might take a few minutes.")
for x, y in points:
    features = sample_all_rasters(x, y, RASTER_FEATURE_MAP)

    if features is None:
        continue  # Point was valid geographically but hit a NoData pixel in a raster

    row = {
        "easting": x,
        "northing": y,
        "point_id": point_id,
        **features
    }

    data.append(row)
    point_id += 1

 Sampling raster data for each point... This might take a few minutes.


In [17]:

#  FORMAT & SAVE CSV
df = pd.DataFrame(data)

# Force exact column order for clean ML input
csv_columns_order = [
    "easting", "northing", "point_id", "distance_to_river", "aspect", "dem", 
    "flow_accumulation", "twi", "slope", "flood", "rainfall", "drainage_density", 
    "ext_rainfall", "lulc", "soil"
]


In [18]:
# Only keep columns that were successfully sampled
available_columns = [col for col in csv_columns_order if col in df.columns]
df = df[available_columns]

# Save output
output_name = "mahanadi_10k_2.csv"
df.to_csv(output_name, index=False)

print(f"Dataset '{output_name}' generated successfully with {len(df)} valid points!")
print(df.head())

Dataset 'mahanadi_10k_2.csv' generated successfully with 9755 valid points!
         easting      northing  point_id  distance_to_river      aspect  \
0  133952.230146  2.249638e+06         0       10993.270508  177.989243   
1  206784.841335  2.468185e+06         1        8457.074219  317.784546   
2  459671.933327  2.241227e+06         2        3313.879883   57.226437   
3   44328.737440  2.570529e+06         3        9742.982422  152.312103   
4   60077.196604  2.480489e+06         4        2313.114746   43.902390   

          dem  flow_accumulation        twi         slope  flood     rainfall  \
0  183.269821                 18  10.043777  1.200029e+00      0  1714.680084   
1  282.669434                  1   8.483146  2.028161e+00      0  1561.825018   
2    0.000346                  6  12.700654  9.576796e-07      0  1767.091847   
3  538.788696                  2   9.058740  4.337976e+00      0  1424.827479   
4  298.437531                  2   8.486233  7.265088e-01      0  18

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

#  1. SET YOUR PATHS HERE
RASTER_DIR = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\data\rasters" # <-- Update to your actual raster folder
BASIN_SHP_PATH = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\data\basin_boudary\mahanadi_basin.shp"

# 2. RASTER FEATURE MAP
RASTER_FEATURE_MAP = {
    "distance_to_river": os.path.join(RASTER_DIR, "distance_to_river.tif"),
    "aspect": os.path.join(RASTER_DIR, "aspect.tif"),
    "dem": os.path.join(RASTER_DIR, "dem_30.tif"),
    "flow_accumulation": os.path.join(RASTER_DIR, "flow_acc.tif"),
    "twi": os.path.join(RASTER_DIR, "TWI.tif"),
    "slope": os.path.join(RASTER_DIR, "slope_30.tif"),
    "rainfall": os.path.join(RASTER_DIR, "rainfall_30m_f.tif"),
    "drainage_density": os.path.join(RASTER_DIR, "drainage_density.tif"), 
    "ext_rainfall": os.path.join(RASTER_DIR, "ext_rainfall.tif"),
    "lulc": os.path.join(RASTER_DIR, "lulc_30m.tif"),
    "soil": os.path.join(RASTER_DIR, "soil_clay_30m.tif"),
    
    #  YOUR NEW INDEPENDENT SATELLITE FLOOD MAP
    "flood": os.path.join(RASTER_DIR, "Mahanadi_Flood_Map_Final.tif")  
}

#  3. LOAD & ALIGN SHAPEFILE (Fix CRS)
print(" Loading Basin Shapefile...")
basin_gdf = gpd.read_file(BASIN_SHP_PATH)

print(" Checking Raster CRS...")
# Open DEM just to get the correct UTM projection
with rasterio.open(RASTER_FEATURE_MAP["dem"]) as src:
    raster_crs = src.crs

# Reproject Shapefile to match Raster CRS (Crucial Step!)
if basin_gdf.crs != raster_crs:
    print(f" Reprojecting basin from {basin_gdf.crs} to {raster_crs}...")
    basin_gdf = basin_gdf.to_crs(raster_crs)

# Combine multi-polygons into a single geometry for checking
basin_geom = basin_gdf.geometry.unary_union
minx, miny, maxx, maxy = basin_geom.bounds

#  4. GENERATE POINTS STRICTLY INSIDE BASIN
points = []
target_points = 10000

print(f" Generating {target_points} points strictly INSIDE the Mahanadi boundary...")
while len(points) < target_points:
    # Create random point inside the bounding box
    x = np.random.uniform(minx, maxx)
    y = np.random.uniform(miny, maxy)
    p = Point(x, y)
    
    # MAGIC STEP: Check if point is actually inside the irregular polygon shape
    if basin_geom.contains(p):
        points.append((x, y))

print("Points generated successfully!")

#  5. RASTER SAMPLING FUNCTION
def sample_all_rasters(x, y, rasters):
    values = {}
    for name, path in rasters.items():
        if not os.path.exists(path):
            print(f" Warning: Missing {name} at {path}")
            continue 
            
        with rasterio.open(path) as src:
            val = list(src.sample([(x, y)]))[0][0]
            
            # Skip invalid pixels (NoData values, outside actual raster data)
            if not np.isfinite(val) or val == src.nodata:
                return None  
                
            values[name] = val
            
    return values

#  6. BUILD DATASET
data = []
point_id = 0

print("⏳ Sampling raster data for each point... This might take a few minutes.")
for x, y in points:
    features = sample_all_rasters(x, y, RASTER_FEATURE_MAP)

    if features is None:
        continue  # Point was valid geographically but hit a NoData pixel in a raster

    row = {
        "easting": x,
        "northing": y,
        "point_id": point_id,
        **features
    }

    data.append(row)
    point_id += 1

#  7. FORMAT & SAVE CSV
df = pd.DataFrame(data)

# Force exact column order for clean ML input
csv_columns_order = [
    "easting", "northing", "point_id", "distance_to_river", "aspect", "dem", 
    "flow_accumulation", "twi", "slope", "flood", "rainfall", "drainage_density", 
    "ext_rainfall", "lulc", "soil"
]

# Only keep columns that were successfully sampled
available_columns = [col for col in csv_columns_order if col in df.columns]
df = df[available_columns]

# Save output
output_name = "mahanadi_flood_training_data_Sentinel1_10k.csv"
df.to_csv(output_name, index=False)

print(f" BOOM! Dataset '{output_name}' generated successfully with {len(df)} valid points!")
print(df.head())

In [31]:
import rasterio
import numpy as np

path_jk = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\data\temp_data\fixed_slope.tif"
with rasterio.open(path_jk) as src:
    data = src.read(1)
    data = data[data != src.nodata]

    print("Mean:", np.mean(data))
    print("Max:", np.max(data))

Mean: nan
Max: nan


In [32]:
import rasterio
import numpy as np

with rasterio.open(path_jk) as src:
    data = src.read(1)

    # Remove nodata
    if src.nodata is not None:
        data = data[data != src.nodata]

    # Remove NaN + inf
    data = data[np.isfinite(data)]

    print("Count:", len(data))
    print("Mean:", np.mean(data))
    print("Max:", np.max(data))

Count: 350033364
Mean: 1.8784151
Max: 86.858536


In [33]:
with rasterio.open(path_jk) as src:
    data = src.read(1)

    if src.nodata is not None:
        data = data[data != src.nodata]

    data = data[np.isfinite(data)]

    print("Min:", np.min(data))
    print("Mean:", np.mean(data))
    print("Median:", np.median(data))
    print("95th percentile:", np.percentile(data, 95))
    print("99th percentile:", np.percentile(data, 99))
    print("Max:", np.max(data))

Min: 0.0
Mean: 1.8784151
Median: 1.0022253e-06
95th percentile: 11.127708
99th percentile: 25.349722
Max: 86.858536


In [34]:
flodd_acc = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\data\temp_data\flow_acc_30m.tif"
with rasterio.open(flodd_acc) as src:
    print(src.crs)
    print(src.res)

EPSG:32645
(30.0, 30.0)


In [36]:
path_ky = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\data\Flood_data_final\dem_30.tif"
with rasterio.open(path_ky) as dem:
    print("DEM bounds:", dem.bounds)

with rasterio.open(flodd_acc) as fa:
    print("Flow bounds:", fa.bounds)

DEM bounds: BoundingBox(left=-191076.69514340186, bottom=2130778.5796351717, right=480803.30485659814, top=2621158.5796351717)
Flow bounds: BoundingBox(left=-191170.03107411636, bottom=2130711.9619857045, right=480889.96892588364, top=2621181.9619857045)
